# Retraining the facial emotion ViT

Runs on a free Colab or Kaggle T4. Produces `vit-emotion-v3.onnx`, a
drop-in replacement for `vit-emotion-v2.onnx` in the deployment folder.

## Why retrain at all

The deployed model scores **67.7% accuracy / 66.5% macro-F1** on seven
classes. That is the signature of FER2013, and it is close to the
ceiling *for that dataset* — FER2013's own labels are noisy enough that
human agreement sits around 65±5%. Architecture is not the bottleneck.
The labels are.

So this notebook changes the data, not the model:

1. **Better labels.** FER+ re-annotated the *same* FER2013 images with
   10 crowd workers each. Training against the vote distribution
   instead of a single noisy argmax is the single largest win available,
   and it costs nothing but a different loss function.
2. **Alignment-matched preprocessing.** The deployment now feeds the
   model an eye-line-aligned crop (`face_pipeline.similarity_transform`).
   Training on crops produced by that *same* transform removes the
   train/test geometry mismatch instead of only patching it at
   inference.
3. **Webcam-domain augmentation.** FER2013 is 48×48 grayscale. A webcam
   frame is colour, JPEG-compressed, unevenly lit and often
   motion-blurred. Augmenting toward that distribution is what closes
   the lab-to-live gap.

The class head stays at **7 outputs in the deployment's exact order**,
so nothing downstream has to change.

## 0. Environment

In [ ]:
!pip -q install "transformers>=4.40" "datasets>=3.0" onnx onnxruntime timm

import os, sys, math, json, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Training will take hours. "
          "In Colab: Runtime > Change runtime type > T4 GPU.")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## 1. The label space

Frozen by the exported ONNX classifier head. Changing the order here
without re-exporting would silently mislabel every prediction the
deployment makes, so it is asserted rather than assumed.

Note FER+ adds a `contempt` class (and `unknown` / `NF` columns). The
deployment contract is seven classes, so contempt is **dropped** rather
than merged — folding it into `angry` or `disgust` would teach the model
a category boundary that does not exist in the label set it has to
predict.

In [ ]:
SHARED_LABELS = ("angry", "disgust", "fear", "happy", "neutral", "sad", "surprise")
NUM_CLASSES = len(SHARED_LABELS)
LABEL2ID = {name: i for i, name in enumerate(SHARED_LABELS)}

# Must match vit-emotion-v2-deployment/emotion_config.json exactly.
assert SHARED_LABELS == ("angry","disgust","fear","happy","neutral","sad","surprise")
print({i: n for i, n in enumerate(SHARED_LABELS)})

## 2. Data

Two routes. **A** is the better experiment; **B** needs no credentials
and is the sensible default if you just want to see the pipeline run.

| | Route A — FER+ | Route B — RAF-DB |
|---|---|---|
| Source | FER2013 CSV (Kaggle) + FER+ labels (GitHub) | HuggingFace, no auth |
| Labels | 10 annotators per image, soft | single label |
| Images | 48×48 grayscale | 100×100 colour, real-world |
| Auth | Kaggle API token | none |

Route A is what the "soft labels" argument above is about. Route B
still benefits from alignment-matched preprocessing and webcam
augmentation, and its images are closer to real-world conditions to
begin with.

Set `ROUTE` below.

In [ ]:
ROUTE = "B"   # "A" = FER+ (soft labels, needs Kaggle), "B" = RAF-DB (no auth)

### Route A — FER+ over FER2013

`fer2013new.csv` (the FER+ labels) is row-aligned with the original
`fer2013.csv`. The alignment is positional, so the notebook asserts the
row counts match before using it — a reshuffled mirror of FER2013 would
otherwise pair every image with someone else's votes and quietly
produce a model trained on noise.

To get `fer2013.csv`: upload your `kaggle.json` when prompted, or drop
the CSV into the working directory yourself.

In [ ]:
FERPLUS_URL = "https://raw.githubusercontent.com/microsoft/FERPlus/master/fer2013new.csv"

# FER+ column order in fer2013new.csv, after the two leading columns.
FERPLUS_COLUMNS = ["neutral", "happiness", "surprise", "sadness", "anger",
                   "disgust", "fear", "contempt", "unknown", "NF"]

FERPLUS_TO_SHARED = {
    "neutral": "neutral", "happiness": "happy", "surprise": "surprise",
    "sadness": "sad", "anger": "angry", "disgust": "disgust", "fear": "fear",
    # contempt / unknown / NF are deliberately absent - see the note above.
}

def load_ferplus():
    import pandas as pd, urllib.request

    if not Path("fer2013.csv").exists():
        try:
            from google.colab import files
            print("Upload kaggle.json (Kaggle > Account > Create New API Token)")
            files.upload()
            os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
            os.replace("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
            os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        except Exception as e:
            raise SystemExit(
                "fer2013.csv not found and Kaggle upload unavailable.\n"
                "Download 'deadskull7/fer2013' manually, or use ROUTE = 'B'."
            ) from e
        !pip -q install kaggle
        !kaggle datasets download -d deadskull7/fer2013 -f fer2013.csv --force
        !unzip -o -q fer2013.csv.zip 2>/dev/null || true

    urllib.request.urlretrieve(FERPLUS_URL, "fer2013new.csv")

    fer = pd.read_csv("fer2013.csv")
    plus = pd.read_csv("fer2013new.csv")

    assert len(fer) == len(plus), (
        f"FER2013 has {len(fer)} rows but FER+ has {len(plus)}. "
        "The FER+ labels are aligned POSITIONALLY - a reshuffled mirror of "
        "FER2013 would pair every image with the wrong votes. Get the "
        "original CSV."
    )

    votes = plus[FERPLUS_COLUMNS].to_numpy(dtype=np.float32)

    # Keep only the seven deployment classes, then drop rows whose mass
    # now lives entirely in the discarded columns.
    keep_idx = [FERPLUS_COLUMNS.index(c) for c in FERPLUS_TO_SHARED]
    ordered = [LABEL2ID[FERPLUS_TO_SHARED[FERPLUS_COLUMNS[i]]] for i in keep_idx]

    soft = np.zeros((len(fer), NUM_CLASSES), dtype=np.float32)
    for col, target in zip(keep_idx, ordered):
        soft[:, target] += votes[:, col]

    total = soft.sum(axis=1)
    usable = total > 0
    soft[usable] /= total[usable, None]

    images = np.stack([
        np.array(p.split(), dtype=np.uint8).reshape(48, 48)
        for p in fer["pixels"]
    ])

    split = fer["Usage"].to_numpy()

    print(f"{usable.sum()} of {len(fer)} rows usable "
          f"({(~usable).sum()} were entirely contempt/unknown/NF)")

    return images[usable], soft[usable], split[usable]

### Route B — RAF-DB

Already in the deployment's class order, 100×100 colour, downloads
without credentials. Single-label, so the soft-label loss below reduces
to ordinary cross-entropy with label smoothing.

In [ ]:
RAFDB_ID = "deanngkl/raf-db-7emotions"
RAFDB_LABELS = ("anger","disgust","fear","happiness","neutral","sadness","surprise")
RAFDB_TO_SHARED = {"anger":"angry","disgust":"disgust","fear":"fear",
                   "happiness":"happy","neutral":"neutral","sadness":"sad",
                   "surprise":"surprise"}

def load_rafdb():
    from datasets import load_dataset
    ds = load_dataset(RAFDB_ID, split="train")

    names = tuple(ds.features["label"].names)
    assert names == RAFDB_LABELS, f"upstream label order changed: {names}"

    images = np.stack([np.array(r["image"].convert("RGB")) for r in ds])
    hard = np.array([LABEL2ID[RAFDB_TO_SHARED[names[r["label"]]]] for r in ds])

    # One-hot with a little smoothing, so the same soft-label loss and
    # the same calibration story apply to both routes.
    soft = np.full((len(hard), NUM_CLASSES), 0.02 / (NUM_CLASSES - 1), np.float32)
    soft[np.arange(len(hard)), hard] = 0.98

    rng = np.random.default_rng(SEED)
    order = rng.permutation(len(hard))
    split = np.array(["Training"] * len(hard), dtype=object)
    split[order[: int(0.10 * len(hard))]] = "PublicTest"
    split[order[int(0.10 * len(hard)) : int(0.20 * len(hard))]] = "PrivateTest"

    return images, soft, split

In [ ]:
images, soft_labels, split = load_ferplus() if ROUTE == "A" else load_rafdb()

print("images       :", images.shape, images.dtype)
print("soft labels  :", soft_labels.shape)
for name in ("Training", "PublicTest", "PrivateTest"):
    print(f"  {name:<12} {(split == name).sum():>6}")

hard = soft_labels.argmax(axis=1)
print("\nmajority-vote class balance:")
for i, name in enumerate(SHARED_LABELS):
    print(f"  {name:<9} {(hard == i).sum():>6}  {(hard == i).mean()*100:5.1f}%")

## 3. Alignment-matched preprocessing

This is the step that pairs with the inference-side fix. The deployment
detects a face, then warps it so the eye line is horizontal, the
inter-ocular distance is a fixed fraction of the output width, and the
eyes sit at a fixed height. If training crops are framed differently,
the model spends capacity on a geometry it will never see live.

The constants are copied from `face_pipeline.py` and **must stay in
sync with it** — they define the contract between training and
deployment.

FER2013 images are already tightly cropped and roughly centred, so
running a detector over them adds little; the canonical re-framing
below reproduces the same geometry directly from the known crop
convention. When training on full frames instead, swap in the real
detector.

In [ ]:
import cv2

# --- copied from vit-emotion-v2-deployment/face_pipeline.py ---
EYE_DISTANCE_RATIO = 0.42
EYE_Y_RATIO = 0.40
IMAGE_SIZE = 224

def similarity_transform(eye_left, eye_right, size=IMAGE_SIZE):
    """Identical to face_pipeline.similarity_transform."""
    dst_l = np.array([(0.5 - EYE_DISTANCE_RATIO/2)*size, EYE_Y_RATIO*size], np.float64)
    dst_r = np.array([(0.5 + EYE_DISTANCE_RATIO/2)*size, EYE_Y_RATIO*size], np.float64)
    src_l = np.asarray(eye_left, np.float64); src_r = np.asarray(eye_right, np.float64)

    sv, dv = src_r - src_l, dst_r - dst_l
    slen = float(np.linalg.norm(sv))
    if slen < 1e-3:
        raise ValueError("degenerate eye line")

    scale = float(np.linalg.norm(dv)) / slen
    angle = math.atan2(dv[1], dv[0]) - math.atan2(sv[1], sv[0])
    ca, sa = scale*math.cos(angle), scale*math.sin(angle)

    return np.array([[ca, -sa, dst_l[0] - (ca*src_l[0] - sa*src_l[1])],
                     [sa,  ca, dst_l[1] - (sa*src_l[0] + ca*src_l[1])]], np.float32)

# Where the eyes sit in a conventional tight face crop, as a fraction of
# the crop. Used to place pseudo-landmarks on already-cropped datasets.
CROP_EYE_X = (0.30, 0.70)
CROP_EYE_Y = 0.38

def canonicalise(img):
    """Re-frame an already-cropped face to the deployment's geometry."""
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    h, w = img.shape[:2]
    eye_l = np.float32([CROP_EYE_X[0]*w, CROP_EYE_Y*h])
    eye_r = np.float32([CROP_EYE_X[1]*w, CROP_EYE_Y*h])
    m = similarity_transform(eye_l, eye_r)
    return cv2.warpAffine(img, m, (IMAGE_SIZE, IMAGE_SIZE),
                          flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

# Sanity: the transform must land the eyes exactly where intended.
_m = similarity_transform(np.float32([100,100]), np.float32([160,120]))
_p = cv2.transform(np.float32([[[100,100],[160,120]]]), _m)[0]
assert abs(_p[0][1] - _p[1][1]) < 1e-3, "roll not corrected"
assert abs(_p[0][1] - EYE_Y_RATIO*IMAGE_SIZE) < 1e-3
print("alignment transform verified")

## 4. Webcam-domain augmentation

Each item is a distortion a real camera introduces and the lab dataset
does not. Applied at training time only.

`grayscale` is included deliberately even for the colour route: it
stops the model keying on skin tone, which is both a fairness concern
and a generalisation one.

In [ ]:
MEAN = np.array([0.5, 0.5, 0.5], np.float32)   # matches emotion_config.json
STD  = np.array([0.5, 0.5, 0.5], np.float32)

def augment(img, rng):
    if rng.random() < 0.5:
        img = cv2.flip(img, 1)

    # Small residual geometry: alignment is never perfect live.
    if rng.random() < 0.7:
        angle = rng.uniform(-12, 12)
        scale = rng.uniform(0.90, 1.10)
        tx, ty = rng.uniform(-0.04, 0.04, 2) * IMAGE_SIZE
        m = cv2.getRotationMatrix2D((IMAGE_SIZE/2, IMAGE_SIZE/2), angle, scale)
        m[:, 2] += (tx, ty)
        img = cv2.warpAffine(img, m, (IMAGE_SIZE, IMAGE_SIZE),
                             borderMode=cv2.BORDER_REPLICATE)

    if rng.random() < 0.7:
        img = cv2.convertScaleAbs(img, alpha=rng.uniform(0.75, 1.30),
                                        beta=rng.uniform(-30, 30))

    if rng.random() < 0.25:
        k = int(rng.choice([3, 5]))
        img = cv2.GaussianBlur(img, (k, k), 0)

    if rng.random() < 0.35:            # JPEG artefacts
        q = int(rng.integers(35, 85))
        ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, q])
        if ok:
            img = cv2.imdecode(buf, cv2.IMREAD_COLOR)

    if rng.random() < 0.20:            # sensor noise
        img = cv2.add(img, rng.normal(0, 6, img.shape).astype(np.int16),
                      dtype=cv2.CV_8U)

    if rng.random() < 0.15:            # ignore skin tone
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)

    if rng.random() < 0.10:            # random occlusion (hand, mic, hair)
        s = int(rng.integers(30, 70))
        x, y = rng.integers(0, IMAGE_SIZE - s, 2)
        img[y:y+s, x:x+s] = int(rng.integers(0, 255))

    return img

def to_tensor(img):
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return torch.from_numpy(((rgb - MEAN) / STD).transpose(2, 0, 1))

In [ ]:
from torch.utils.data import Dataset, DataLoader

class FaceDataset(Dataset):
    def __init__(self, images, soft, train):
        self.images, self.soft, self.train = images, soft, train

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        rng = np.random.default_rng((SEED * 1000003 + i) % (2**32))
        img = canonicalise(self.images[i])
        if self.train:
            img = augment(img, rng)
        return to_tensor(img), torch.from_numpy(self.soft[i])

train_ds = FaceDataset(images[split == "Training"],   soft_labels[split == "Training"],   True)
val_ds   = FaceDataset(images[split == "PublicTest"], soft_labels[split == "PublicTest"], False)
test_ds  = FaceDataset(images[split == "PrivateTest"],soft_labels[split == "PrivateTest"],False)

BATCH = 64 if DEVICE == "cuda" else 16

train_loader = DataLoader(train_ds, BATCH, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  BATCH, shuffle=False, num_workers=2)

print(f"train {len(train_ds)}  val {len(val_ds)}  test {len(test_ds)}")

In [ ]:
# Eyeball a batch before committing an hour of GPU to it. Every image
# should be a face, upright, with the eyes on the same horizontal line.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for ax, i in zip(axes.flat, np.random.default_rng(0).integers(0, len(train_ds), 12)):
    t, s = train_ds[int(i)]
    img = (t.numpy().transpose(1, 2, 0) * STD + MEAN).clip(0, 1)
    ax.imshow(img); ax.axis("off")
    ax.set_title(SHARED_LABELS[int(s.argmax())], fontsize=9)
plt.suptitle("Aligned + augmented training crops")
plt.tight_layout(); plt.show()

## 5. Model and loss

### Soft-label KL divergence

The headline change. An image where 6 of 10 annotators said `sad` and 4
said `neutral` is not a `sad` image — it is genuinely ambiguous, and
training it as hard `sad` forces the model to be confident about
something humans were not. Matching the vote distribution instead lets
the model express the same uncertainty, which improves both accuracy
and calibration.

`alpha` mixes in class-balanced hard cross-entropy, because pure KL on
a long-tailed set still lets `disgust` and `fear` be ignored.

In [ ]:
from transformers import ViTForImageClassification

MODEL_ID = "google/vit-base-patch16-224-in21k"

model = ViTForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_CLASSES,
    id2label={i: n for i, n in enumerate(SHARED_LABELS)},
    label2id=LABEL2ID,
).to(DEVICE)

print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

# Inverse-frequency weights over the majority-vote labels.
counts = np.bincount(soft_labels[split == "Training"].argmax(1), minlength=NUM_CLASSES)
counts = np.maximum(counts, 1)
class_w = torch.tensor(counts.sum() / (NUM_CLASSES * counts), dtype=torch.float32, device=DEVICE)
print("class weights:", {n: round(float(w), 2) for n, w in zip(SHARED_LABELS, class_w)})

def soft_loss(logits, target, alpha=0.7):
    """alpha * KL(target || pred) + (1-alpha) * weighted CE on the argmax."""
    kl = F.kl_div(F.log_softmax(logits, -1), target, reduction="batchmean")
    ce = F.cross_entropy(logits, target.argmax(-1), weight=class_w)
    return alpha * kl + (1 - alpha) * ce

## 6. Train

In [ ]:
EPOCHS = 6
LR = 3e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = len(train_loader) * EPOCHS
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, total_steps=steps, pct_start=0.1)
scaler = torch.amp.GradScaler(enabled=(DEVICE == "cuda"))

@torch.no_grad()
def evaluate(loader):
    model.eval()
    all_logits, all_targets = [], []
    for x, y in loader:
        with torch.autocast(DEVICE, enabled=(DEVICE == "cuda")):
            out = model(pixel_values=x.to(DEVICE)).logits
        all_logits.append(out.float().cpu()); all_targets.append(y)
    logits = torch.cat(all_logits); targets = torch.cat(all_targets)
    pred, true = logits.argmax(1).numpy(), targets.argmax(1).numpy()

    f1 = []
    for c in range(NUM_CLASSES):
        tp = ((pred == c) & (true == c)).sum()
        fp = ((pred == c) & (true != c)).sum()
        fn = ((pred != c) & (true == c)).sum()
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f1.append(2*p*r/(p+r) if p+r else 0.0)

    return (pred == true).mean(), float(np.mean(f1)), np.array(f1), logits, true

best_f1, history = -1.0, []

for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0; t0 = time.time()

    for step, (x, y) in enumerate(train_loader, 1):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE, enabled=(DEVICE == "cuda")):
            loss = soft_loss(model(pixel_values=x.to(DEVICE)).logits, y.to(DEVICE))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item()
        if step % 50 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_loader)} "
                  f"loss {running/step:.4f} {(time.time()-t0)/step:.2f}s/step", flush=True)

    acc, f1, per_class, _, _ = evaluate(val_loader)
    history.append({"epoch": epoch, "loss": running/len(train_loader),
                    "val_acc": float(acc), "val_macro_f1": f1})
    print(f"epoch {epoch}  val acc {acc*100:.2f}%  macro-F1 {f1*100:.2f}%")

    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained("best-vit-emotion-v3")
        print(f"  saved (best macro-F1 {f1*100:.2f}%)")

print(f"\nbest validation macro-F1 {best_f1*100:.2f}%")

## 7. Test and calibrate

Temperature is fitted on validation and stored in the config the
deployment reads. The fusion module pools the two modalities by their
log-probabilities, so an overconfident face model would win arguments
it has not earned — calibrating it is what keeps the comparison fair.

In [ ]:
model = ViTForImageClassification.from_pretrained("best-vit-emotion-v3").to(DEVICE)

acc, f1, per_class, val_logits, val_true = evaluate(val_loader)
t_acc, t_f1, t_per_class, _, _ = evaluate(test_loader)

print(f"TEST  accuracy {t_acc*100:.2f}%   macro-F1 {t_f1*100:.2f}%")
print("\nper-class F1 (%)")
for n, v in zip(SHARED_LABELS, t_per_class):
    print(f"  {n:<9} {v*100:6.2f}")

print("\nbaseline (deployed v2): 67.69% accuracy / 66.46% macro-F1")

# --- temperature scaling, fitted on validation ---
def ece(probs, true, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    correct = (pred == true).astype(float); e = 0.0
    for lo, hi in zip(np.linspace(0,1,bins+1)[:-1], np.linspace(0,1,bins+1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def softmax_np(z):
    z = z - z.max(-1, keepdims=True); e = np.exp(z); return e / e.sum(-1, keepdims=True)

lg = val_logits.numpy()
nll = lambda T: float(-np.log(np.clip(
    softmax_np(lg / T)[np.arange(len(val_true)), val_true], 1e-12, 1)).mean())

g = (5**0.5 - 1) / 2; a, b = 0.05, 10.0
c, d = b - g*(b-a), a + g*(b-a); fc, fd = nll(c), nll(d)
for _ in range(60):
    if fc < fd: b, d, fd = d, c, fc; c = b - g*(b-a); fc = nll(c)
    else:       a, c, fc = c, d, fd; d = a + g*(b-a); fd = nll(d)
    if abs(b-a) < 1e-4: break
TEMPERATURE = (a + b) / 2

print(f"\ntemperature {TEMPERATURE:.4f}")
print(f"validation ECE {ece(softmax_np(lg), val_true):.4f} -> "
      f"{ece(softmax_np(lg/TEMPERATURE), val_true):.4f}")

# Monotone: it must not change a single prediction.
assert np.array_equal(lg.argmax(1), (lg/TEMPERATURE).argmax(1))

## 8. Export to ONNX

The signature must match `vit-emotion-v2.onnx` exactly — input
`pixel_values` of shape `[N,3,224,224]`, output `logits` of shape
`[N,7]`, dynamic batch — or `face_pipeline.EmotionClassifier` will not
load it. The batch axis must stay dynamic because flip-TTA sends two
views in one call.

In [ ]:
import onnx, onnxruntime as ort

model.eval().cpu()

class Wrapper(nn.Module):
    """Unwrap the HF output object so ONNX sees a plain logits tensor."""
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, pixel_values): return self.m(pixel_values=pixel_values).logits

dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)

torch.onnx.export(
    Wrapper(model), dummy, "vit-emotion-v3.onnx",
    input_names=["pixel_values"], output_names=["logits"],
    dynamic_axes={"pixel_values": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17, do_constant_folding=True,
)

onnx.checker.check_model(onnx.load("vit-emotion-v3.onnx"))

sess = ort.InferenceSession("vit-emotion-v3.onnx", providers=["CPUExecutionProvider"])
print("input :", sess.get_inputs()[0].name, sess.get_inputs()[0].shape)
print("output:", sess.get_outputs()[0].name, sess.get_outputs()[0].shape)

# Parity check: ONNX must agree with PyTorch, and batch-2 must work
# because that is exactly what flip-TTA sends.
with torch.no_grad():
    torch_out = Wrapper(model)(dummy).numpy()
onnx_out = sess.run(None, {"pixel_values": dummy.numpy()})[0]

print(f"\nmax |torch - onnx| = {np.abs(torch_out - onnx_out).max():.2e}")
assert np.allclose(torch_out, onnx_out, atol=1e-4), "ONNX export diverged"
assert onnx_out.shape == (2, NUM_CLASSES)
print("export verified")

config = {
    "image_size": IMAGE_SIZE,
    "mean": MEAN.tolist(),
    "std": STD.tolist(),
    "temperature": round(float(TEMPERATURE), 4),
    "id2label": {str(i): n for i, n in enumerate(SHARED_LABELS)},
    "_provenance": {
        "route": ROUTE,
        "base_model": MODEL_ID,
        "epochs": EPOCHS,
        "test_accuracy": round(float(t_acc), 4),
        "test_macro_f1": round(float(t_f1), 4),
        "per_class_f1": {n: round(float(v), 4) for n, v in zip(SHARED_LABELS, t_per_class)},
    },
}
Path("emotion_config.json").write_text(json.dumps(config, indent=4))
print(json.dumps(config["_provenance"], indent=2))

## 9. Install into the deployment

Download both files and `vit-emotion-v3.onnx.data` if one was produced
(large models are exported with external weights — the two files must
travel together).

Then, in `vit-emotion-v2-deployment/`:

1. Copy `vit-emotion-v3.onnx` (+ `.data`) into the folder.
2. Point `server.py`'s `ONNX_PATH` at it.
3. Replace `emotion_config.json` with the one exported above — it
   carries the fitted `temperature`.
4. Copy `_provenance.per_class_f1` into `reliability.json` under
   `face.per_class_f1`, so fusion weights each class by the *new*
   model's measured competence rather than v2's.
5. Re-run `python test_deployment.py` — `test_shared_space_matches_the_exported_model`
   will catch a label-order mistake immediately.
6. Re-run `python eval_face.py --data eval_data/webcam` for the
   before/after table.

In [ ]:
try:
    from google.colab import files
    files.download("vit-emotion-v3.onnx")
    if Path("vit-emotion-v3.onnx.data").exists():
        files.download("vit-emotion-v3.onnx.data")
    files.download("emotion_config.json")
except ImportError:
    print("Not on Colab - the files are in the working directory:")
    for f in ("vit-emotion-v3.onnx", "vit-emotion-v3.onnx.data", "emotion_config.json"):
        if Path(f).exists():
            print(f"  {f}  ({Path(f).stat().st_size/1e6:.1f} MB)")

## Notes for the writeup

* **What changed and what did not.** Same architecture, same class
  head, same export signature. Only the labels, the crop geometry and
  the augmentation differ. That makes the comparison against v2 a clean
  one — any gain is attributable to data, not capacity.
* **Soft labels are the honest part of the story.** Report the FER+
  vote-distribution loss as the primary change and quote the
  human-agreement figure alongside the accuracy, so a reader can see
  why 85% on FER+ and 68% on FER2013 are not contradictory numbers.
* **Datasets worth mentioning but not used here.** RAF-DB (~30k
  real-world images, licence request) and AffectNet (~400k, academic
  request) both beat FER2013 on realism and both are gated. Say so
  rather than implying FER2013 was the only option.
* **The augmentation is a hypothesis, not a guarantee.** If the webcam
  gap does not close, the next thing to check is whether the deployment
  and training crops really are identical — dump a few of each and
  overlay them.